## 2. Chuẩn bị môi trường và tải dữ liệu

In [2]:
!pip uninstall -y dataproc-spark-connect
!pip install -q pyspark==3.5.1
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet -O taxi_data.parquet
!ls -lh taxi_data.parquet

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0
-rw-r--r-- 1 root root 48M Mar 21  2024 taxi_data.parquet


## Phần A: Khởi tạo và khám phá dữ liệu

### Câu 1. Khởi tạo SparkSession, đọc Parquet, in schema và 5 dòng đầu

In [16]:
from pyspark.sql import SparkSession

MSSV = "201230161"
spark = (SparkSession.builder
         .appName(f"ThucHanh_Tuan1_{MSSV}")
         .master("local[*]")
         .getOrCreate())

df = spark.read.parquet("taxi_data.parquet")

df.printSchema()
df.show(5)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

### Câu 2. Đếm tổng số chuyến và số chuyến bất thường

In [19]:
from pyspark.sql.functions import col

tong_chuyen = df.count()
so_bat_thuong = df.filter((col("trip_distance") <= 0) | (col("total_amount") <= 0)).count()
so_chuyen_duoi_5 = df.filter(col("trip_distance") < 5).count()
so_sach = df.count()

print(f"Tổng số chuyến đi trong dữ liệu gốc : {tong_chuyen:,}")
print(f"Số chuyến bất thường (distance<=0 hoặc total<=0): {so_bat_thuong:,}")
print(f"Số chuyến đi có quãng đường dưới 5 dặm: {so_chuyen_duoi_5:,}")
print(f"Chiếm {so_chuyen_duoi_5 / so_sach * 100:.2f}% tổng số chuyến sạch")
print(f"Tỷ lệ bất thường: {so_bat_thuong / tong_chuyen * 100:.2f}%")

Tổng số chuyến đi trong dữ liệu gốc : 2,964,624
Số chuyến bất thường (distance<=0 hoặc total<=0): 92,603
Số chuyến đi có quãng đường dưới 5 dặm: 2,500,268
Chiếm 84.34% tổng số chuyến sạch
Tỷ lệ bất thường: 3.12%


### Câu 3. Tạo `clean_df` và tính tỷ lệ bản ghi sạch

In [21]:
clean_df = df.filter(
    (col("trip_distance") > 0) &
    (col("total_amount") > 0) &
    (col("passenger_count") > 0)
).cache()

so_sach = clean_df.count()
print(f"Số bản ghi sạch còn lại: {so_sach:,}")
print(f"Tỷ lệ so với ban đầu   : {so_sach / tong_chuyen * 100:.2f}%")

Số bản ghi sạch còn lại: 2,724,017
Tỷ lệ so với ban đầu   : 91.88%


## Phần B: Xử lý dữ liệu phân tán với RDD

### Câu 4. RDD quãng đường: tổng, trung bình, lớn nhất, nhỏ nhất

In [22]:
distance_rdd = clean_df.select("trip_distance").rdd.map(lambda row: float(row[0]))

tong_qd = distance_rdd.sum()
tb_qd   = distance_rdd.mean()
max_qd  = distance_rdd.max()
min_qd  = distance_rdd.min()

print(f"(a) Tổng quãng đường          : {tong_qd:,.2f} dặm")
print(f"(b) Quãng đường trung bình    : {tb_qd:.3f} dặm/chuyến")
print(f"(c) Dài nhất: {max_qd:,.2f} dặm | Ngắn nhất: {min_qd:.2f} dặm")

(a) Tổng quãng đường          : 8,996,415.04 dặm
(b) Quãng đường trung bình    : 3.303 dặm/chuyến
(c) Dài nhất: 15,400.32 dặm | Ngắn nhất: 0.01 dặm


### Câu 5. MapReduce: số chuyến theo số lượng hành khách (giảm dần)

In [7]:
from operator import add

passenger_rdd = clean_df.select("passenger_count").rdd.map(lambda row: int(row[0]))

ket_qua = (passenger_rdd
           .map(lambda p: (p, 1))          # map: (số khách, 1)
           .reduceByKey(add)               # reduce: cộng dồn theo số khách
           .sortBy(lambda kv: kv[1], ascending=False))
print("Số khách | Số chuyến")
for so_khach, so_chuyen in ket_qua.collect():
    print(f"{so_khach:>8} | {so_chuyen:,}")

Số khách | Số chuyến
       1 | 2,134,798
       2 | 395,121
       3 | 88,973
       4 | 49,597
       5 | 33,307
       6 | 22,178
       8 | 37
       7 | 5
       9 | 1


### Câu 6. Đếm số chuyến có tiền boa từ $10 trở lên

In [25]:
import pandas as pd

tip_passenger_rdd = clean_df.select("tip_amount", "passenger_count").rdd.map(lambda row: (row[0], row[1]))

so_chuyen_boa_cao = tip_passenger_rdd.filter(
    lambda x: x[0] is not None and x[0] >= 10 and x[1] is not None and x[1] < 5
).count()

print(f"Số chuyến có tiền boa >= $10 và dưới 5 khách: {so_chuyen_boa_cao:,}")

Số chuyến có tiền boa >= $10 và dưới 5 khách: 180,281


## Phần C: Phân tích, gom nhóm và xuất kết quả với DataFrame

### Câu 7. Thống kê theo hình thức thanh toán

In [9]:
from pyspark.sql import functions as F

pay_df = (clean_df.groupBy("payment_type")
          .agg(F.count("*").alias("tong_chuyen"),
               F.round(F.avg("fare_amount"), 2).alias("cuoc_tb"),
               F.round(F.avg("tip_amount"), 2).alias("boa_tb"))
          .orderBy("payment_type"))
pay_df.show()

# NHẬN XÉT:
# - Thanh toán thẻ (1) có tiền boa trung bình cao, còn tiền mặt (2) gần như bằng 0.
# - Nguyên nhân: tip_amount chỉ được hệ thống TỰ ĐỘNG ghi nhận khi khách trả bằng thẻ
#   (khách chọn mức boa trên máy POS). Khi trả tiền mặt, tiền boa đưa trực tiếp cho tài xế
#   và KHÔNG được nhập vào hệ thống, nên dữ liệu ghi 0 -> chênh lệch lớn chủ yếu do cách ghi nhận,
#   không hẳn do khách trả tiền mặt boa ít hơn.
# - Cước phí trung bình giữa hai nhóm khá gần nhau, nên sự khác biệt nằm ở tiền boa chứ không phải giá cước.

+------------+-----------+-------+------+
|payment_type|tong_chuyen|cuoc_tb|boa_tb|
+------------+-----------+-------+------+
|           1|    2273594|   18.4|  4.16|
|           2|     417921|  18.65|   0.0|
|           3|       9854|  17.24|  0.01|
|           4|      22648|  19.75|   0.0|
+------------+-----------+-------+------+



### Câu 8. Phân tích khung giờ cao điểm

In [10]:
from pyspark.sql.functions import hour

hour_df = (clean_df
           .withColumn("gio_don", hour(col("tpep_pickup_datetime")))
           .groupBy("gio_don")
           .agg(F.count("*").alias("tong_chuyen"),
                F.round(F.sum("total_amount"), 2).alias("tong_doanh_thu")))

print("Thống kê theo từng giờ trong ngày (0h – 23h):")
hour_df.orderBy("gio_don").show(24)

top5_df = hour_df.orderBy(col("tong_chuyen").desc()).limit(5)
print("Top 5 khung giờ có nhiều chuyến nhất:")
top5_df.show()

Thống kê theo từng giờ trong ngày (0h – 23h):
+-------+-----------+--------------+
|gio_don|tong_chuyen|tong_doanh_thu|
+-------+-----------+--------------+
|      0|      69196|    2000362.75|
|      1|      45383|    1174736.36|
|      2|      31941|     786132.17|
|      3|      20551|     541926.12|
|      4|      12790|     416111.96|
|      5|      15630|     592224.22|
|      6|      35583|    1077604.17|
|      7|      74321|    1964678.82|
|      8|     105128|    2669719.97|
|      9|     119401|     3087145.4|
|     10|     130046|    3389435.72|
|     11|     141273|    3598904.11|
|     12|     153620|    3954249.39|
|     13|     159097|    4216675.73|
|     14|     171284|    4739588.25|
|     15|     176985|    4851392.87|
|     16|     178419|    5318194.75|
|     17|     191275|    5399653.24|
|     18|     195927|    5279208.58|
|     19|     170600|    4747569.05|
|     20|     149275|    4093747.13|
|     21|     148398|    4104520.37|
|     22|     130542|    3739

### Bổ sung: Phân tích tháng cao điểm

In [26]:
from pyspark.sql.functions import month

month_df = (clean_df
            .withColumn("thang_don", month(col("tpep_pickup_datetime")))
            .groupBy("thang_don")
            .agg(F.count("*").alias("tong_chuyen"),
                 F.round(F.sum("total_amount"), 2).alias("tong_doanh_thu")))

print("Thống kê theo từng tháng trong dữ liệu:")
month_df.orderBy("thang_don").show()


Thống kê theo từng tháng trong dữ liệu:
+---------+-----------+--------------+
|thang_don|tong_chuyen|tong_doanh_thu|
+---------+-----------+--------------+
|        1|    2724003| 7.466496284E7|
|        2|          3|         90.47|
|       12|         11|        235.12|
+---------+-----------+--------------+



### Câu 9. Đo hiệu năng: `toPandas().sum()` vs `agg(sum)`

In [11]:
import time
from pyspark.sql.functions import sum as spark_sum

t1 = time.time()
pdf = df.select("total_amount").toPandas()
tong_pandas = pdf["total_amount"].sum()
t2 = time.time()

t3 = time.time()
tong_spark = df.agg(spark_sum("total_amount")).collect()[0][0]
t4 = time.time()

print(f"Pandas : tổng = {tong_pandas:,.2f} | thời gian = {t2 - t1:.3f} giây")
print(f"PySpark: tổng = {tong_spark:,.2f} | thời gian = {t4 - t3:.3f} giây")

# NHẬN XÉT: PySpark tính phân tán ngay trên các partition rồi chỉ trả về 1 số,
# còn toPandas() phải kéo toàn bộ dữ liệu về bộ nhớ của driver nên chậm và tốn RAM hơn.
# (Kết quả thời gian có thể thay đổi giữa các lần chạy.)

Pandas : tổng = 79,456,384.28 | thời gian = 21.369 giây
PySpark: tổng = 79,456,384.28 | thời gian = 0.585 giây


### Câu 10. Xuất Top 5 khung giờ ra CSV

In [12]:
duong_dan = "/content/ketqua_tuan1"

(top5_df.coalesce(1)
        .write.mode("overwrite")
        .option("header", True)
        .csv(duong_dan))

import glob, pandas as pd
file_csv = glob.glob(duong_dan + "/part-*.csv")[0]
print("Đã lưu:", file_csv)
pd.read_csv(file_csv)

Đã lưu: /content/ketqua_tuan1/part-00000-b305ad5b-50db-4f19-a58f-d9128e15d965-c000.csv


,gio_don,tong_chuyen,tong_doanh_thu
0,18,195927,5279208.58
1,17,191275,5399653.24
2,16,178419,5318194.75
3,15,176985,4851392.87
4,14,171284,4739588.25


In [13]:
spark.stop()